# Deploy Gemma 4 (12B) on AMD via Ollama

This notebook installs Ollama on AMD ROCm hardware to run `gemma4:12b` (Instruction-Tuned) efficiently.
It also sets up Cloudflare Tunnel (or SSH tunnel) to expose the Ollama API (which is OpenAI-compatible at port `11434`) publicly so your Fuenzer Sports backend can reach it.

### Step 1: Install Ollama
We use the official Ollama Linux installer script, which automatically detects your AMD GPU and sets up ROCm acceleration.

In [ ]:
# Download and install Ollama (requires root/sudo access in the notebook environment)
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start Ollama Server
We start the Ollama server process in the background using Python's `subprocess` so it doesn't block the notebook.

In [ ]:
import subprocess

# Start Ollama server in background and redirect logs to ollama.log
proc = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
print("✅ Ollama server successfully started in background on port 11434.")

### Step 3: Pull Gemma 4 Model
We pull `gemma4:12b`, which fits perfectly on a single 48GB AMD GPU.

In [ ]:
# Pull Gemma 4 12B model from Ollama registry
!ollama pull gemma4:12b

### Step 4: Verify and Test Model
Check that the model is loaded and run a test query to verify GPU acceleration works.

In [ ]:
# List installed models
!ollama list

# Test model response
!ollama run gemma4:12b "Say hello from Fuenzer Sports!"

### Step 5: Start Tunnel
This will output a public URL. Copy this URL and set it as `LOCAL_MODEL_BASE_URL` in your backend's `.env` file (append `/v1` to it, e.g. `https://xxx.trycloudflare.com/v1`).

*(Choose either Cloudflare Tunnel or the SSH fallback tunnel)*

#### Option A: Cloudflare Tunnel

In [ ]:
# Download cloudflared if not present, then start tunnel to Ollama port 11434
!curl -kL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared || \
 wget -q --no-check-certificate https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Expose Ollama server
!cloudflared tunnel --url http://localhost:11434

#### Option B: Fallback SSH Tunnel (Run this if Cloudflare fails to connect)

In [ ]:
import subprocess
import time

proc_tunnel = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-R", "80:localhost:11434", "nokey@localhost.run"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

# Read output to get the public URL
for line in proc_tunnel.stdout:
    if ".localhost.run" in line:
        print(f"🔗 Public URL (SSH Fallback): {line.strip()}")
        break